In [2]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import sys, random, subprocess, importlib.metadata as metadata
from pathlib import Path
import numpy as np
import torch

SEED = 266
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
roots = [Path.cwd(), *Path.cwd().parents,
         Path("/content/data266-lab2-team41"),
         Path("/kaggle/working/data266-lab2-team41")]
ROOT = next((p for p in roots if (p / "src/p2_partner_baseline.py").is_file()), None)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip() if ROOT else "unavailable"
print("Git commit:", commit)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("SEED:", SEED)
assert ROOT is not None and commit != "unavailable", "Clone the repository first; see eval/PART2_YUYAO_BASELINE.md."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
assert torch.cuda.is_available() and torch.cuda.device_count() == 1, "Use one visible T4/P100 GPU."
for package in ("diffusers", "transformers", "peft", "accelerate", "datasets", "huggingface-hub"):
    try: print(package, metadata.version(package))
    except metadata.PackageNotFoundError: print(package, "not installed yet")

Git commit: ef31e7bdd3088c10e59b44f259c928e299da4a4f
Python: 3.13.16
PyTorch: 2.14.1+cu130
GPU: Tesla T4
SEED: 266
diffusers 0.42.1
transformers 5.22.1
peft 0.21.2
accelerate 1.15.0
datasets 4.8.5
huggingface-hub 1.33.0


# Part 2: Yuyao Ding's completed partner working baseline

**Completed on a real single Colab Tesla T4.** This copy retains the supplied executed code cells and numeric outputs. Initial preparation used commit `ef31e7b`; actual training used the recorded logging fix at `9e60390`. The first cell and preparation manifest report their original pre-fix state, not a retroactively assigned commit.

The private original also contains a clone cell and a one-off logging-recovery cell. Those temporary cells are removed here. The regular training cell's output explicitly reuses the completed record from that recovery execution; the private original and saved log contain the actual 600-step training. Original execution counts are retained. Three embedded image previews, including original training-image panels, are omitted from this public copy. Numeric outputs remain saved; complete visuals and weights remain in the private archive for approved hosting.

[Measured results, source hashes, failure/recovery evidence and limitations](../../eval/part2/results/yuyao_t4_20261115T034917Z/README.md).

Team Rule 2 requires the partner to implement and run the complete pipeline at least as a working baseline. This notebook covers data preparation, one real rank-4 LoRA training run, paired base/adapter generation, KID/CLIP/LPIPS, training cost, safety checks and nearest-training-image inspection.

Pratiksha Kaushik's `p2/p2_v4.ipynb`, two 3,000-step runs and chosen rank-32 model remain the team submission. This shorter **600-step**, effective-batch-4 baseline records Yuyao's own execution. It is not a replacement final model, a new holdout, or evidence that 600 steps is a separately specified course minimum. Record actual participation and the real knowledge-transfer meeting in CONTRIBUTIONS.md; do not claim execution before running it.

Open this notebook from the cloned repository in a **fresh single Colab T4 or Kaggle T4/P100 runtime**. The first cell prints the actual commit, Python/Torch, GPU and seed. Save the executed notebook after the run. No teammate Drive directory is used.

## 1. Prepare dependencies without replacing CUDA/PyTorch

The ML API pins below match the versions printed in our earlier real T4 Part 3 run; the supplied run now records this pipeline on an actual T4. The first code cell reports versions before setup; the saved run manifest records the active versions afterward. The setup constrains existing Torch, torchvision, torchaudio, NumPy and pandas. It removes an optional incompatible torchao installation only in this dedicated cloud GPU session. It never installs Torch or changes a Windows environment.

If the cell asks for a restart because a changed package was already imported, restart once and rerun from cell 1. Hugging Face credentials, if needed, may be supplied as a Colab secret `HF_TOKEN`; they are never printed or written to artifacts.

In [3]:
from packaging.version import Version
from src.p2_partner_baseline import gpu_check
print("Hardware check:", gpu_check())
assert Path("/content").is_dir() or Path("/kaggle").is_dir(), "Use a fresh Colab/Kaggle GPU runtime for setup."
requirements = ROOT / "config/requirements-part2-baseline.txt"
lines = [line.strip() for line in requirements.read_text().splitlines()
         if line.strip() and not line.lstrip().startswith("#")]
changed = []
for line in lines:
    name, target = line.split("==")
    try: current = metadata.version(name)
    except metadata.PackageNotFoundError: current = None
    if current != target: changed.append(name)
protected = {}
for name in ("torch", "torchvision", "torchaudio", "numpy", "pandas"):
    try: protected[name] = metadata.version(name)
    except metadata.PackageNotFoundError: pass
setup_dir = ROOT / ".runtime/part2-setup"
setup_dir.mkdir(parents=True, exist_ok=True)
constraint = setup_dir / "preserve-existing-packages.txt"
constraint.write_text("\n".join(f"{name}=={version}" for name, version in protected.items()) + "\n")
restart_needed = False
try:
    if Version(metadata.version("torchao")) < Version("0.16.0"):
        restart_needed = "torchao" in sys.modules
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
except metadata.PackageNotFoundError:
    pass
if changed:
    loaded = {"diffusers": "diffusers", "transformers": "transformers", "peft": "peft",
              "accelerate": "accelerate", "datasets": "datasets", "huggingface-hub": "huggingface_hub",
              "torchmetrics": "torchmetrics", "torch-fidelity": "torch_fidelity", "Pillow": "PIL",
              "lpips": "lpips", "ImageHash": "imagehash"}
    restart_needed |= any(loaded.get(name, name) in sys.modules for name in changed)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements),
                    "-c", str(constraint)], check=True)
assert all(metadata.version(name) == version for name, version in protected.items()), "Protected package changed."
if restart_needed:
    raise RuntimeError("Setup finished. Restart the session once, then rerun from cell 1.")
for name in ("diffusers", "transformers", "peft", "accelerate", "datasets", "torchmetrics"):
    print(name, metadata.version(name))
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        token = userdata.get("HF_TOKEN")
        if token: os.environ["HF_TOKEN"] = token
        del token
except Exception:
    pass
from diffusers import StableDiffusionPipeline
from peft import LoraConfig
from torchmetrics.image.kid import KernelInceptionDistance
import lpips, datasets, imagehash
print("Dependency imports: PASSED")

Hardware check: {'name': 'Tesla T4', 'total_gib': 14.56317138671875}
diffusers 0.42.1
transformers 5.22.1
peft 0.21.2
accelerate 1.15.0
datasets 3.1.0
torchmetrics 1.6.0
Dependency imports: PASSED


## 2. Freeze this run and its provenance

The committed 20 prompts reuse Pratiksha's previously fixed prompts, with attribution. They are fixed before this new training and remain development prompts. The in-domain captions must actually occur in this run's test split. Base, safety and CLIP model revisions and the dataset revision are resolved once and recorded.

Default artifacts are written under the ignored `.runtime/part2-yuyao/`. To retain a run on a mounted persistent volume, set `DATA266_P2_OUTPUT_DIR` to your own new run folder before this cell. To resume, set it to the same folder; config, prompt and source hashes must match. Never point it at your teammate's output directory. Colab/Kaggle temporary storage may disappear after a session expires, so download the final bundle promptly.

In [4]:
import json
from src.p2_partner_baseline import BaselineConfig, start_run, read, fetch_trainer, sha
CFG = BaselineConfig(train_steps=600)
RUN = start_run(ROOT, output=os.environ.get("DATA266_P2_OUTPUT_DIR"), config=CFG)
trainer = fetch_trainer(RUN)
print("Verified official trainer SHA256:", sha(trainer))
print("Run directory:", RUN)
print(json.dumps(read(RUN / "manifest.json"), indent=2))

Verified official trainer SHA256: 7d67f3ed756a3cb79183153349ebb2d141c999f898da964edb2ea4af5d72d6ab
Run directory: /content/data266-lab2-team41/.runtime/part2-yuyao/20261115T034917Z
{
  "status": "prepared",
  "owner": "Yuyao Ding",
  "interpretation": "Partner working baseline; not the team final comparison or an untouched holdout.",
  "config": {
    "seed": 266,
    "resolution": 512,
    "rank": 4,
    "train_steps": 600,
    "micro_batch": 1,
    "gradient_accumulation": 4,
    "learning_rate": 0.0001,
    "inference_steps": 30,
    "guidance_scale": 7.5
  },
  "gpu": {
    "name": "Tesla T4",
    "total_gib": 14.56317138671875
  },
  "source_commit": "ef31e7bdd3088c10e59b44f259c928e299da4a4f",
  "source_hashes": {
    "src/p2_partner_baseline.py": "ea8056252698856e6575ff8aa68ed89dfb0475b0c9e070f85108f26a5ec30ff7",
    "scripts/p2_train_worker.py": "29bed9c4f61c14cc4735ba55f0eb213caed1caadc346eb3ff06165f83e64722f"
  },
  "prompts_sha256": "4298b0378332c6dc9b2e56eb9c5712b436416daba6

## 3. Prepare the data

We inspect image bytes and captions, remove MD5 and perceptual-hash duplicates, split 90/10 with seed 266, center-crop/resize to 512×512, and screen training images with the SD safety checker. Flagged training images are kept outside the actual training directory and never enter the trainer. The test set is not used for gradients. The printed report is computed, not copied from the teammate's counts.

Dataset limitations: Naruto screenshots and automatically generated captions can overrepresent certain characters and views; a conservative NSFW checker can remove benign anime images. No unrestricted data license is assumed. Use only for non-commercial research under the model/data terms.

In [5]:
from src.p2_partner_baseline import prepare_data
data_report = prepare_data(RUN)
print(json.dumps(data_report, indent=2))

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

data/train-00000-of-00002-12944970063701(…): reconstructing file:   0%|          |  0.00B /  344MB            

data/train-00000-of-00002-12944970063701(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-cefa2f480689f1(…): reconstructing file:   0%|          |  0.00B /  357MB            

... (235 outputs left out)

Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


Training safety filter: 1088/1098


Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


Training safety filter: 1096/1098
Training safety filter: 1098/1098
{
  "dataset": "lambdalabs/naruto-blip-captions",
  "dataset_revision": "1ed13e8ef280bd45e3bbac4cfa8bbd9d64ec9f89",
  "original": 1221,
  "kept_before_split": 1221,
  "removed_ids": {
    "corrupted": [],
    "exact_duplicates": [],
    "near_duplicates": []
  },
  "split_seed": 266,
  "train_before_safety": 1098,
  "train_nsfw_flagged": 240,
  "train_after_safety": 858,
  "test": 123,
  "resolution": 512,
  "license": "Not stated in dataset card; copyrighted Naruto images, non-commercial research only.",
  "train_metadata_sha256": "e831a7eccc663067a6ac0c3c1ed8e864bb29838893b59f473ca2318a6570a5dc",
  "train_ids_sha256": "dc3f16c1dc230677a9361d23469873cc52d917c9ca98b31c922d6c05c02602fb",
  "test_ids_sha256": "50e734882bf5d24cd64c6c020b7fc706b29d8820b912eabc41dee93142ff542b"
}


## 4. Train one rank-4 LoRA

The same frozen UNet attention targets and diffusion-noise objective as the teammate's workflow are implemented using the official Diffusers SD1.5 LoRA trainer. The upstream example is pinned by commit and SHA-256; attribution and source are in the runbook. It trains only the adapter, using fp16 frozen weights, fp32 trainable adapters, gradient checkpointing, micro-batch 1 and accumulation 4. The budget is **600 optimizer steps**, LR 1e-4, cosine schedule, seed 266.

This starts real training. Logs stream below and checkpoints are saved every 100 steps. If training is interrupted, retain the run folder and rerun this cell with the same RUN to resume. A completed run is reused only after verifying its adapter hash. Duration, peak memory and actual saved adapter parameters are measured in the GPU worker, not inferred from another run.

In [8]:
from src.p2_partner_baseline import train_lora
training_stats = train_lora(ROOT, RUN)
print(json.dumps(training_stats, indent=2))

Using this run's completed training record; no new training.
{
  "status": "completed",
  "gpu": "Tesla T4",
  "adapter_sha256": "b40c06bf43e61565c293ce5d1c6c760f17a43e996b6af69339ee33cb50f22f03",
  "trainable_parameters": 797184,
  "rank": 4,
  "learning_rate": 0.0001,
  "requested_steps": 600,
  "micro_batch": 1,
  "gradient_accumulation": 4,
  "effective_batch": 4,
  "wall_seconds_this_invocation": 1186.685664718,
  "cumulative_wall_seconds": 1198.7448562230002,
  "peak_allocated_gib": 3.681016445159912,
  "peak_reserved_gib": 4.11328125,
  "trainer_sha256": "7d67f3ed756a3cb79183153349ebb2d141c999f898da964edb2ea4af5d72d6ab",
  "external_tracking": "disabled: report_to converted to Python None"
}


## 5. Generate a paired base/adapter comparison

Generate 20 prompts × seeds 0 and 1 for each model: 40 base images and 40 images with **your newly trained adapter**. Both use DPM-Solver++, 30 steps, CFG 7.5 and 512×512. The first pass uses a fresh base pipeline without a LoRA. Output safety checks stay enabled; flagged returned images may be blacked out, and their flags are retained. Metrics include the returned safety-filtered images, so do not directly pool these results with a pipeline that scores unfiltered images.

Each image's identity, hash and timing is saved immediately. A restart reuses validated saved images and generates only missing combinations.

In [9]:
from src.p2_partner_baseline import generate
generations = generate(RUN)
print("Images:", len(generations))
print("Flagged:", sum(row["nsfw"] for row in generations))

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Generated 1/80: base_p00_s0.png
Generated 2/80: base_p00_s1.png
Generated 3/80: base_p01_s0.png
Generated 4/80: base_p01_s1.png
Generated 5/80: base_p02_s0.png
Generated 6/80: base_p02_s1.png
Generated 7/80: base_p03_s0.png
Generated 8/80: base_p03_s1.png
Generated 9/80: base_p04_s0.png
Generated 10/80: base_p04_s1.png
Generated 11/80: base_p05_s0.png
Generated 12/80: base_p05_s1.png
Generated 13/80: base_p06_s0.png
Generated 14/80: base_p06_s1.png
Generated 15/80: base_p07_s0.png
Generated 16/80: base_p07_s1.png
Generated 17/80: base_p08_s0.png
Generated 18/80: base_p08_s1.png
Generated 19/80: base_p09_s0.png
Generated 20/80: base_p09_s1.png
Generated 21/80: base_p10_s0.png
Generated 22/80: base_p10_s1.png
Generated 23/80: base_p11_s0.png
Generated 24/80: base_p11_s1.png
Generated 25/80: base_p12_s0.png
Generated 26/80: base_p12_s1.png
Generated 27/80: base_p13_s0.png
Generated 28/80: base_p13_s1.png
Generated 29/80: base_p14_s0.png
Generated 30/80: base_p14_s1.png
Generated 31/80: ba

Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


Generated 39/80: base_p19_s0.png
Generated 40/80: base_p19_s1.png


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


Generated 41/80: yuyao_lora_r4_p00_s0.png
Generated 42/80: yuyao_lora_r4_p00_s1.png
Generated 43/80: yuyao_lora_r4_p01_s0.png
Generated 44/80: yuyao_lora_r4_p01_s1.png
Generated 45/80: yuyao_lora_r4_p02_s0.png
Generated 46/80: yuyao_lora_r4_p02_s1.png
Generated 47/80: yuyao_lora_r4_p03_s0.png
Generated 48/80: yuyao_lora_r4_p03_s1.png
Generated 49/80: yuyao_lora_r4_p04_s0.png
Generated 50/80: yuyao_lora_r4_p04_s1.png
Generated 51/80: yuyao_lora_r4_p05_s0.png
Generated 52/80: yuyao_lora_r4_p05_s1.png
Generated 53/80: yuyao_lora_r4_p06_s0.png
Generated 54/80: yuyao_lora_r4_p06_s1.png
Generated 55/80: yuyao_lora_r4_p07_s0.png
Generated 56/80: yuyao_lora_r4_p07_s1.png
Generated 57/80: yuyao_lora_r4_p08_s0.png
Generated 58/80: yuyao_lora_r4_p08_s1.png
Generated 59/80: yuyao_lora_r4_p09_s0.png
Generated 60/80: yuyao_lora_r4_p09_s1.png
Generated 61/80: yuyao_lora_r4_p10_s0.png
Generated 62/80: yuyao_lora_r4_p10_s1.png
Generated 63/80: yuyao_lora_r4_p11_s0.png
Generated 64/80: yuyao_lora_r4_p11

## 6. Evaluate from your generated artifacts

KID uses all 20 in-domain generated images and the held-out test images, with 100 subsets; it is noisy at this size. CLIP score is 100 × the nonnegative normalized image/text cosine using CLIP ViT-B/16. LPIPS uses AlexNet on each seed pair. Costs are measured training GPU wall-hours and peak allocated VRAM. Separately compute nearest-training-image similarities for 10 adapted in-domain images; similarity is a diagnostic, not proof of memorization or its absence. No human ratings are synthesized.

In [10]:
import pandas as pd
from src.p2_partner_baseline import evaluate
results = evaluate(RUN)
columns = ["model", "images", "kid_in_domain", "kid_std", "clip_score", "lpips",
           "nsfw_flagged", "median_seconds_per_image", "training_gpu_hours",
           "training_peak_allocated_gib", "generation_peak_allocated_gib"]
display(pd.DataFrame(results)[columns])
print("Metrics saved to:", RUN / "results.json")

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

... (10 outputs left out)

Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)  # noqa: B028
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:01<00:00, 69.3MB/s]


,model,images,kid_in_domain,kid_std,clip_score,lpips,nsfw_flagged,median_seconds_per_image,training_gpu_hours,training_peak_allocated_gib,generation_peak_allocated_gib
0,base,40,0.133655,0.012166,31.108441,0.811634,1,4.856718,0.000000,NaN,3.180614
1,yuyao_lora_r4,40,0.090961,0.010269,30.702620,0.792775,0,5.107185,0.332985,3.681016,3.182099


Metrics saved to: /content/data266-lab2-team41/.runtime/part2-yuyao/20261115T034917Z/results.json


## 7. Inspect side-by-side outputs and failure cases

Inspect actual outputs before making claims about domain style, objects, text, or safety. The fixed eight-prompt grid includes in-domain, OOD and compositional prompts. A second grid shows the adapted model's lowest CLIP-score prompts; this ranking is not a human pass/fail judgment. Write your own observations after viewing these images.

In [11]:
from PIL import Image
import matplotlib.pyplot as plt
rows = read(RUN / "generations.json")
lookup = {(r["model"], r["prompt_id"], r["seed"]): r for r in rows}
def show_pairs(ids):
    fig, axes = plt.subplots(len(ids), 2, figsize=(9, 4 * len(ids)), squeeze=False)
    for i, pid in enumerate(ids):
        for j, model in enumerate(("base", "yuyao_lora_r4")):
            row = lookup[(model, pid, 0)]
            axes[i, j].imshow(Image.open(RUN / row["file"]))
            axes[i, j].set_title(f"{model} | p{pid} | NSFW={row['nsfw']}")
            axes[i, j].axis("off")
        axes[i, 0].set_ylabel(lookup[("base", pid, 0)]["prompt"])
    plt.tight_layout(); plt.show()
show_pairs([0, 1, 2, 3, 10, 12, 15, 16])
worst = sorted([r for r in rows if r["model"] == "yuyao_lora_r4" and r["seed"] == 0],
               key=lambda r: r["clip_score"])[:3]
show_pairs([r["prompt_id"] for r in worst])

[Visual preview retained in the private original executed notebook and archive; image payload omitted from this public copy.]

[Visual preview retained in the private original executed notebook and archive; image payload omitted from this public copy.]

In [12]:
neighbors = sorted(read(RUN / "memorization_neighbors.json"),
                   key=lambda r: r["cosine"], reverse=True)
display(pd.DataFrame(neighbors))
fig, axes = plt.subplots(3, 2, figsize=(9, 12))
for i, row in enumerate(neighbors[:3]):
    for j, field in enumerate(("generated_file", "training_file")):
        axes[i, j].imshow(Image.open(RUN / row[field]))
        axes[i, j].set_title(f"{'Generated' if j == 0 else 'Nearest training'} | cosine={row['cosine']:.3f}")
        axes[i, j].axis("off")
plt.tight_layout(); plt.show()

,generated_file,training_file,cosine
0,images/yuyao_lora_r4_p06_s0.png,data/train/00319.png,0.751333
1,images/yuyao_lora_r4_p03_s0.png,data/train/00254.png,0.735096
2,images/yuyao_lora_r4_p02_s0.png,data/train/01185.png,0.716203
3,images/yuyao_lora_r4_p07_s0.png,data/train/00456.png,0.714398
4,images/yuyao_lora_r4_p01_s0.png,data/train/00245.png,0.708259
5,images/yuyao_lora_r4_p05_s0.png,data/train/01136.png,0.686403
6,images/yuyao_lora_r4_p00_s0.png,data/train/01143.png,0.620894
7,images/yuyao_lora_r4_p09_s0.png,data/train/00698.png,0.608897
8,images/yuyao_lora_r4_p08_s0.png,data/train/00893.png,0.574177
9,images/yuyao_lora_r4_p04_s0.png,data/train/00860.png,0.525912


[Visual preview retained in the private original executed notebook and archive; image payload omitted from this public copy.]

## 8. Yuyao's own interpretation and participation record

Complete after inspecting the results. Do not replace these items with invented observations:

- Which examples gained a recognizable domain style, and which did not?
- Give two concrete failure examples and their prompt IDs.
- Discuss whether prompt alignment/diversity changed, including noisy KID and safety-filter effects.
- Inspect the three closest generated/training pairs; do not infer “no memorization” from a threshold alone.
- Explain how the frozen VAE/text encoder, LoRA layers and diffusion-noise loss work.
- Record the real knowledge-transfer meeting date/topic/recording link with Pratiksha.

The supplied executed notebook and matching private artifacts now record the actual T4 pipeline. Yuyao's own interpretation and knowledge-transfer recording still need to be added. This baseline has no new human-preference scores; the team's separate 25-pair, three-rater evaluation remains its own requirement.

## 9. Save the evidence bundle

The private ZIP includes this executed run's metadata, metrics, adapter and generated images, but excludes original training/test images and optimizer checkpoints. Keep the run directory on persistent storage if you may need to resume training. Save/download this notebook itself from Colab after execution; the source notebook file on disk does not automatically receive browser outputs.

Review the bundle before publishing. Original data stays private; adapter weights and generated image files belong on Hugging Face, not in GitHub. Nothing is uploaded automatically.

In [13]:
from zipfile import ZipFile, ZIP_DEFLATED
manifest = read(RUN / "manifest.json")
assert manifest["status"] == "completed"
archive = RUN.parent / f"Part2_Yuyao_Baseline_{RUN.name}.zip"
files = [p for p in RUN.glob("*.json")] + [RUN / "training.log"]
files += list((RUN / "images").glob("*.png"))
files += list((RUN / "adapter").glob("*.safetensors"))
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for path in files:
        assert path.is_file(), path
        bundle.write(path, arcname=str(Path(RUN.name) / path.relative_to(RUN)))
print("Bundle:", archive)
print("Size MiB:", round(archive.stat().st_size / 1024**2, 2))
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))

Bundle: /content/data266-lab2-team41/.runtime/part2-yuyao/Part2_Yuyao_Baseline_20261115T034917Z.zip
Size MiB: 35.91


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>